# Hydrogeological Analysis — Tordera Water Captures (Palafolls)

**Authors:** 
- Carlos Daniel Muñoz Sánchez
- Lorena Larrotta Morales
- Yomery Mercedes

**Context:** Hydrogeological characterization project within the framework of the M.Sc. in Science and Integrated Water Management — Universitat de Barcelona (2024–2026)  
**Study Area:** Lower Tordera alluvial aquifer, municipality of Palafolls (Barcelona, Spain)  

---

## Objectives

1. Interpret the **step-drawdown pumping test** of Well A using the **Cooper-Jacob** method to estimate Transmissivity (T) and Storage Coefficient (S).
2. Evaluate the **hydraulic efficiency** of the well through head loss analysis (Jacob's method).
3. Calculate the **protection perimeters** of the capture using the analytical **Wyssling** method for three time horizons.

---

## Conceptual Framework

### Cooper-Jacob Method
For unsteady flow conditions in a confined aquifer, Cooper & Jacob (1946) simplified Theis's solution when the parameter $u = r^2S / (4Tt) < 0.05$. The drawdown in a piezometer can be expressed as:

$$s = \frac{2.3Q}{4\pi T} \log_{10}\left(/frac{2.25Tt}{r^2 S}\right)$$

Transmissivity is obtained from the slope $\Delta s$ of the straight line on a semilogarithmic scale:

$$T = \frac{2.3Q}{4\pi \Delta s}$$

### Well Efficiency (Jacob, 1947)
The total drawdown in the pumped well is broken down into linear losses (laminar flow in the aquifer) and quadratic losses (near-well turbulence):

$$\frac{s_w}{Q} = B + CQ$$

Efficiency is defined as the ratio of theoretical drawdown (linear losses) to total drawdown:

$$E_w = \frac{BQ}{BQ + CQ^2} \times 100$$

### Protection Perimeters — Wyssling
Wyssling's method calculates the capture zone of a well in an aquifer with uniform regional flow, defining an ellipse whose extension upstream ($L_+$) and downstream ($L_-$) depends on the transit time:

$$X_0 = \frac{Q}{2\pi T i}, \quad L_\pm = \frac{X_0}{2}\left(\sqrt{1+\alpha} \pm 1\right), \quad \alpha = \frac{2\pi n_e b v_0 t}{Q}$$


In [ ]:
# ── LIBRARIES ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print('✓ Libraries loaded successfully')

---
## 1. Data Loading and Preparation

In [ ]:
# ── DATA LOADING ─────────────────────────────────────────────────────────────
# The original file is a CSV with ';' separator and comma decimals (European format)
df_raw = pd.read_csv(
    'data/palafolls_pouA.csv',   # adjust path if necessary
    sep=';',
    encoding='utf-8-sig'
)

# Clean column names
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.rename(columns={'Q ': 'Q'})

# Convert numerical columns (comma -> dot)
num_cols = ['t', 'S-alfa', 'S- B', 'S - C']
for col in num_cols:
    df_raw[col] = (
        df_raw[col].astype(str)
        .str.replace(',', '.', regex=False)
        .replace({'nan': np.nan, 'None': np.nan})
        .astype(float)
    )

# Time in minutes
df_raw['t_min'] = df_raw['t'] / 60

# Separate pumping phase and recovery
df_bombeo = df_raw[df_raw['Q'].isin(['Q1', 'Q2', 'Q3'])].copy()
df_rec     = df_raw[df_raw['Q'] == 'Recuperacion'].copy()

print(f'Total records: {len(df_raw)}')
print(f'  → Pumping:   {len(df_bombeo)} rows')
print(f'  → Recovery:  {len(df_rec)} rows')
df_bombeo.head()

In [ ]:
# ── TEST PARAMETERS ──────────────────────────────────────────────────────

# Flow rates per step (m³/s)
Q_vals = {
    'Q1': 265  / 3600,   # 265 m³/h → m³/s
    'Q2': 327.2 / 3600,
    'Q3': 377  / 3600,
}

# Observation wells and distances to pumped well
pozos = {
    'alfa': {'r': None,  'col': 'S-alfa', 'label': 'Well A (pumped)', 'color': '#e63946'},
    'B':    {'r': 91,    'col': 'S- B',   'label': 'Piezometer B (91 m)', 'color': '#2a9d8f'},
    'C':    {'r': 55,    'col': 'S - C',  'label': 'Piezometer C (55 m)', 'color': '#457b9d'},
}

print('Test configuration:')
print(f'  Step Q1: {265:.0f} m³/h ({Q_vals["Q1"]*1000:.2f} L/s)')
print(f'  Step Q2: {327.2:.0f} m³/h ({Q_vals["Q2"]*1000:.2f} L/s)')
print(f'  Step Q3: {377:.0f} m³/h ({Q_vals["Q3"]*1000:.2f} L/s)')
print(f'  Distances: B = 91 m | C = 55 m from pumped well')

---
## 2. Complete Step-Drawdown Test Visualization

In [ ]:
# ── COMPLETE DRAWDOWN CURVE ─────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

q_labels = ['Q1', 'Q2', 'Q3']
q_display = ['Q1 = 265 m³/h', 'Q2 = 327.2 m³/h', 'Q3 = 377 m³/h']
colors = {'alfa': '#e63946', 'B': '#2a9d8f', 'C': '#457b9d'}

for ax, q_label, q_disp in zip(axes, q_labels, q_display):
    data_q = df_bombeo[df_bombeo['Q'] == q_label]
    
    for pname, pinfo in pozos.items():
        sub = data_q[data_q[pinfo['col']].notna()].sort_values('t_min')
        if len(sub) > 2:
            ax.semilogx(
                sub['t_min'], sub[pinfo['col']],
                'o-', color=pinfo['color'], ms=4, lw=1.5,
                label=pinfo['label']
            )
    
    ax.set_xlabel('Time (min)', fontsize=10)
    ax.set_ylabel('Drawdown s (m)', fontsize=10)
    ax.set_title(f'Step {q_disp}', fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    ax.invert_yaxis()

fig.suptitle('Step-Drawdown Pumping Test — Well A, Palafolls (Lower Tordera)\nDrawdown curves per step', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('output/01_curvas_descenso.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: output/01_curvas_descenso.png')

---
## 3. Cooper-Jacob Method — T and S Estimation

In [ ]:
# ── COOPER-JACOB FUNCTION ───────────────────────────────────────────────────────
def cooper_jacob(t_min, s, Q, r=None):
    """
    Cooper-Jacob linear fit on a semilogarithmic scale.
    
    Parameters
    ----------
    t_min : array  Time (minutes)
    s     : array  Drawdown (m)
    Q     : float  Flow rate (m³/s)
    r     : float  Well-piezometer distance (m); None for pumped well
    
    Returns
    -------
    T_dia : Transmissivity (m²/day)
    S     : Storage coefficient (-), only if r is known
    slope, intercept, R2
    """
    logt = np.log10(t_min)
    slope, intercept, r_val, _, _ = linregress(logt, s)
    
    T_seg  = (2.3 * Q) / (4 * np.pi * slope)       # m²/s
    T_dia  = T_seg * 86400                         # m²/day
    
    if r is not None and T_seg > 0:
        t0 = 10 ** (-intercept / slope)            # minutes
        t0_seg = t0 * 60
        S_coef = (4 * T_seg * t0_seg) / (r ** 2)
    else:
        S_coef = None
    
    return T_dia, S_coef, slope, intercept, r_val**2

print('✓ Cooper-Jacob function defined')

In [ ]:
# ── APPLY COOPER-JACOB TO ALL STEPS AND PIEZOMETERS ─────────────────
resultados_cj = []

# Apply only to piezometers B and C (with known distance) for
# S calculation; well alfa yields T without S (no r defined).
pozos_cj = {
    'B': {'r': 91,  'col': 'S- B'},
    'C': {'r': 55,  'col': 'S - C'},
    'alfa (well)': {'r': None, 'col': 'S-alfa'},
}

for q_label, Q in Q_vals.items():
    for pname, pinfo in pozos_cj.items():
        data = (
            df_bombeo[
                (df_bombeo['Q'] == q_label) &
                (df_bombeo[pinfo['col']].notna()) &
                (df_bombeo['t_min'] > 0)
            ].sort_values('t_min')
        )
        if len(data) < 5:
            continue

        # Quasi-steady state interval (20%–70% of data)
        n = len(data)
        i1, i2 = int(0.2 * n), int(0.7 * n)
        t_sel = data['t_min'].iloc[i1:i2]
        s_sel = data[pinfo['col']].iloc[i1:i2]

        if len(t_sel) < 3:
            continue

        T_dia, S_coef, slope, intercept, R2 = cooper_jacob(
            t_sel.values, s_sel.values, Q, pinfo['r']
        )

        if T_dia <= 0 or not np.isfinite(T_dia):
            continue

        resultados_cj.append({
            'Step': q_label,
            'Q (m³/h)': Q * 3600,
            'Point': pname,
            'r (m)': pinfo['r'] if pinfo['r'] else '—',
            'T (m²/day)': round(T_dia, 1),
            'S (-)': f"{S_coef:.2e}" if S_coef else '—',
            'R²': round(R2, 3),
            '_slope': slope, '_intercept': intercept,
            '_t': data['t_min'].values, '_s': data[pinfo['col']].values,
            '_t_sel': t_sel.values, '_s_sel': s_sel.values,
        })

df_cj = pd.DataFrame(resultados_cj)
print('COOPER-JACOB RESULTS\n')
print(df_cj[['Step','Q (m³/h)','Point','r (m)','T (m²/day)','S (-)','R²']].to_string(index=False))

In [ ]:
# ── COOPER-JACOB PLOTS ─────────────────────────────────────────────────────
# Filter only well alfa (most complete) for main visualization
rows_alfa = df_cj[df_cj['Point'] == 'alfa (well)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colores_q = {'Q1': '#2a9d8f', 'Q2': '#e9c46a', 'Q3': '#e63946'}

for ax, (_, row) in zip(axes, rows_alfa.iterrows()):
    t_all, s_all = row['_t'], row['_s']
    t_sel, s_sel = row['_t_sel'], row['_s_sel']
    slope, intercept = row['_slope'], row['_intercept']
    color = colores_q[row['Step']]
    
    # All data
    ax.semilogx(t_all, s_all, 'o', color=color, ms=5, alpha=0.5, label='Observed data')
    # Points used in fit
    ax.semilogx(t_sel, s_sel, 's', color=color, ms=6, label='Fit points')
    # Cooper-Jacob line
    t_line = np.linspace(t_sel.min(), t_sel.max(), 100)
    s_line = slope * np.log10(t_line) + intercept
    ax.semilogx(t_line, s_line, '-k', lw=2, label='C-J Fit')
    
    ax.invert_yaxis()
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Drawdown s (m)')
    ax.set_title(f"{row['Step']} — Q = {row['Q (m³/h)']:.0f} m³/h", fontweight='bold')
    
    txt = f"T = {row['T (m²/day)']} m²/day\nR² = {row['R²']}"
    ax.text(0.04, 0.05, txt, transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.legend(fontsize=8)

fig.suptitle('Cooper-Jacob Method — Well A (pumped)\nSemilogarithmic fit per step', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/02_cooper_jacob.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: output/02_cooper_jacob.png')

---
## 4. Well Efficiency (Jacob's Method)

In [ ]:
# ── WELL EFFICIENCY ────────────────────────────────────────────────────────
# Stabilized drawdowns at the end of each step in the pumped well (m)
# (values at the end of each step in column S-alfa)
s_estabilizados = {
    'Q1': 5.41,    # m — stabilized drawdown Q1=265 m³/h
    'Q2': 7.08,    # m — Q2=327.2 m³/h
    'Q3': 8.67,    # m — Q3=377 m³/h
}

Q_eff = np.array([265, 327.2, 377])      # m³/h
s_eff = np.array([5.41, 7.08, 8.67])     # m

# s/Q ratio
s_Q = s_eff / Q_eff

# Linear fit: s/Q = B + C·Q
slope_eff, intercept_eff, r_eff, _, _ = linregress(Q_eff, s_Q)
B = intercept_eff    # linear losses (m·h/m³)
C = slope_eff        # quadratic losses (m·h²/m⁶)

# Efficiency per step
eficiencia = (B * Q_eff) / (B * Q_eff + C * Q_eff**2) * 100

df_eff = pd.DataFrame({
    'Q (m³/h)': Q_eff,
    's (m)': s_eff,
    's/Q': s_Q.round(6),
    'Linear losses BQ (m)': (B * Q_eff).round(3),
    'Non-linear losses CQ² (m)': (C * Q_eff**2).round(3),
    'Efficiency (%)': eficiencia.round(1)
})

print('WELL EFFICIENCY — JACOB METHOD\n')
print(df_eff.to_string(index=False))
print(f'\nB = {B:.5f} m·h/m³')
print(f'C = {C:.6f} m·h²/m⁶')
print(f'Fit R² = {r_eff**2:.3f}')

In [ ]:
# ── EFFICIENCY PLOT ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# — Plot 1: s/Q vs Q (Jacob diagnostic) —
Q_fit = np.linspace(240, 400, 200)
axes[0].plot(Q_eff, s_Q, 'o', color='#e63946', ms=8, zorder=5, label='Observed data')
axes[0].plot(Q_fit, intercept_eff + slope_eff * Q_fit, '-', color='#1d3557', lw=2, label=f'Linear fit (R²={r_eff**2:.3f})')
axes[0].set_xlabel('Flow rate Q (m³/h)')
axes[0].set_ylabel('s / Q (m·h/m³)')
axes[0].set_title('Efficiency diagnostic\n(Jacob, 1947)', fontweight='bold')
axes[0].legend()

# — Plot 2: Drawdown breakdown —
x = np.arange(len(Q_eff))
w = 0.35
bars1 = axes[1].bar(x - w/2, B * Q_eff, w, label='Linear losses (BQ)', color='#2a9d8f', alpha=0.85)
bars2 = axes[1].bar(x + w/2, C * Q_eff**2, w, label='Non-linear losses (CQ²)', color='#e76f51', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Q{i+1}\n{q:.0f} m³/h' for i, q in enumerate(Q_eff)])
axes[1].set_ylabel('Drawdown (m)')
axes[1].set_title('Head loss components\nPer step', fontweight='bold')
axes[1].legend()

# Annotate efficiency
for i, eff in enumerate(eficiencia):
    axes[1].text(i, max(B*Q_eff[i], C*Q_eff[i]**2) + 0.05, 
                 f'Eff={eff:.0f}%', ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Efficiency Analysis — Well A, Palafolls', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/03_eficiencia_pozo.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: output/03_eficiencia_pozo.png')

---
## 5. Protection Perimeters — Wyssling Method

In [ ]:
# ── HYDROGEOLOGICAL PARAMETERS (from Cooper-Jacob analysis and aquifer data) ─
params = {
    'Q':   1080.0,    # m³/day — design concession flow rate
    'T':   8685.7,    # m²/day — mean transmissivity (Cooper-Jacob)
    'ne':  0.20,      # effective porosity (Baix Tordera alluvial aquifer)
    'b':   13.0,      # m — saturated aquifer thickness
    'i':   0.003,     # regional hydraulic gradient
    'pozo_x': 479732.0,  # UTM 31N X (m)
    'pozo_y': 4614252.0, # UTM 31N Y (m)
}

# Protection time horizons
tiempos = {
    '60 days':  60,
    '100 days': 100,
    '5 years':  1825,  # 365 × 5
}

# Derived calculations
K  = params['T'] / params['b']                         # Hydraulic conductivity (m/day)
q0 = K * params['i']                                   # Specific discharge (m/day)
v0 = q0 / params['ne']                                   # Pore velocity (m/day)
Xo = params['Q'] / (2 * np.pi * params['T'] * params['i'])  # Radius of capture / stagnation point (m)

print('AQUIFER PARAMETERS')
print(f'  K  = {K:.1f} m/day')
print(f'  q0 = {q0:.4f} m/day (specific discharge)')
print(f'  v0 = {v0:.4f} m/day (effective pore velocity)')
print(f'  X0 = {Xo:.1f} m (capture radius / stagnation point)')

In [ ]:
# ── WYSSLING FUNCTION ───────────────────────────────────────────────────────────
def wyssling_perimeter(t_dias, Q, T, ne, b, i, n_pts=300):
    """
    Calculates protection perimeter according to Wyssling.
    
    Returns x, y arrays (coordinates relative to well) and a dict with
    geometric parameters of the perimeter.
    """
    Xo    = Q / (2 * np.pi * T * i)
    K     = T / b
    q0    = K * i
    v0    = q0 / ne
    alpha = 2 * np.pi * ne * b * v0 * t_dias / Q

    L_plus  = Xo * (np.sqrt(1 + alpha) + 1) / 2
    L_minus = Xo * (np.sqrt(1 + alpha) - 1) / 2
    B_ancho = 2 * np.sqrt(Xo * L_plus)

    # Shifted ellipse
    a       = (L_plus + L_minus) / 2
    b_ell   = B_ancho / 2
    cx      = -L_minus

    theta = np.linspace(0, 2 * np.pi, n_pts)
    x = cx + a * np.cos(theta)
    y = b_ell * np.sin(theta)

    geom = {
        'Xo (m)': round(Xo, 1),
        'L+ upstream (m)': round(L_plus, 1),
        'L- downstream (m)': round(L_minus, 1),
        'Max width B (m)': round(B_ancho, 1),
        'alpha': round(alpha, 4),
    }
    return x, y, geom

# Calculate and display parameters per horizon
print('GEOMETRIC PARAMETERS — WYSSLING\n')
for label, t in tiempos.items():
    _, _, geom = wyssling_perimeter(t, **{k: params[k] for k in ['Q','T','ne','b','i']})
    print(f'Horizon {label}:')
    for k, v in geom.items():
        print(f'  {k}: {v}')
    print()

In [ ]:
# ── PROTECTION PERIMETERS PLOT ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colores_wyss = {'60 days': '#2a9d8f', '100 days': '#e9c46a', '5 years': '#e63946'}
lw_wyss     = {'60 days': 2,         '100 days': 2,         '5 years': 2.5}
ls_wyss     = {'60 days': '--',        '100 days': '-.',        '5 years': '-'}

for ax_idx, ax in enumerate(axes):
    # Pumped well
    if ax_idx == 0:  # relative coords
        ax.plot(0, 0, 'r*', ms=14, zorder=10, label='Well A', markeredgecolor='black', markeredgewidth=0.5)
    else:            # UTM coords
        ax.plot(params['pozo_x'], params['pozo_y'], 'r*', ms=14, zorder=10,
                label='Well A (479732, 4614252)', markeredgecolor='black', markeredgewidth=0.5)

    # Regional flow arrow
    arrow_len = 300 if ax_idx == 1 else 250
    ox = params['pozo_x'] - 800 if ax_idx == 1 else -800
    oy = params['pozo_y'] if ax_idx == 1 else 0
    ax.annotate('', xy=(ox + arrow_len, oy), xytext=(ox, oy),
                arrowprops=dict(arrowstyle='->', color='#1d3557', lw=1.5))
    ax.text(ox, oy + 80, 'Regional flow', fontsize=8, color='#1d3557')

    for label, t in tiempos.items():
        x_w, y_w, geom = wyssling_perimeter(t, **{k: params[k] for k in ['Q','T','ne','b','i']})
        if ax_idx == 1:
            x_w = x_w + params['pozo_x']
            y_w = y_w + params['pozo_y']
        ax.plot(x_w, y_w, color=colores_wyss[label], lw=lw_wyss[label],
                ls=ls_wyss[label], label=f'Wyssling {label}')
        # Annotate extension
        idx_max = np.argmin(x_w) if ax_idx == 1 else np.argmin(x_w)
        ax.annotate(
            f"{geom['L+ upstream (m)']:.0f} m",
            xy=(x_w[idx_max], y_w[idx_max]),
            xytext=(x_w[idx_max] - 50, y_w[idx_max] + 40),
            fontsize=7, color=colores_wyss[label],
            arrowprops=dict(arrowstyle='->', color=colores_wyss[label], lw=0.8)
        )

    ax.set_aspect('equal')
    ax.legend(fontsize=9, loc='upper right')
    if ax_idx == 0:
        ax.set_xlabel('Relative E-W distance (m)')
        ax.set_ylabel('Relative N-S distance (m)')
        ax.set_title('Perimeters relative to well', fontweight='bold')
    else:
        ax.set_xlabel('UTM X (m) — Zone 31N')
        ax.set_ylabel('UTM Y (m) — Zone 31N')
        ax.set_title('Perimeters in UTM 31N coordinates', fontweight='bold')
        ax.ticklabel_format(style='plain', axis='both')

fig.suptitle('Protection Perimeters — Wyssling Method\nWell A (403A21), Lower Tordera — Palafolls',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/04_perimetros_wyssling.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: output/04_perimetros_wyssling.png')

---
## 6. Summary of Results

In [ ]:
# ── SUMMARY TABLE ──────────────────────────────────────────────────────────────
print('=' * 65)
print('   HYDROGEOLOGICAL SUMMARY — TORDERA CAPTURES (PALAFOLLS)')
print('=' * 65)

print('\n▶ TRANSMISSIVITY (Cooper-Jacob, pumped well):')
for _, row in df_cj[df_cj['Point'] == 'alfa (well)'].iterrows():
    print(f"   {row['Step']} — T = {row['T (m²/day)']} m²/day  (R² = {row['R²']})")

print(f'   → Adopted value: T = {params["T"]} m²/day')

print('\n▶ WELL EFFICIENCY (Jacob):')
for i, q in enumerate(Q_eff):
    print(f'   Q = {q:.0f} m³/h → Efficiency = {eficiencia[i]:.1f}%')

print('\n▶ PROTECTION PERIMETERS (Wyssling):')
for label, t in tiempos.items():
    _, _, geom = wyssling_perimeter(t, **{k: params[k] for k in ['Q','T','ne','b','i']})
    print(f'   {label:12s} → L+ = {geom["L+ upstream (m)"]:7.1f} m | '
          f'L- = {geom["L- downstream (m)"]:6.1f} m | '
          f'B = {geom["Max width B (m)"]:7.1f} m')

print('\n' + '=' * 65)

---
## 7. Discussion and Interpretation

### Transmissivity
The transmissivity values obtained via Cooper-Jacob are consistent with those expected for coarse alluvial aquifers (gravels and sands) in the Lower Tordera area, where T typically ranges between 5,000 and 15,000 m²/day. The good fit of the model (R² > 0.95 in steps with more data) indicates that the applicability conditions of the method are met.

### Well Efficiency
An efficiency > 80% is considered acceptable in water supply wells. The obtained values allow evaluating the structural condition of the well and estimating whether clogging or deterioration of the screening zone exists.

### Protection Perimeters
Wyssling's method is an analytical approximation assuming a homogeneous, isotropic aquifer and uniform regional flow. The calculated perimeters are conservative and represent the capture zone of the well for the time horizons established by current legislation (WFD, Real Decreto 849/1986).

The upstream extension for 5 years (~1,800 m) reflects the high transmissivity of the aquifer and justifies the need to implement protection measures in a wide strip upstream of the well.

---

## References

- Cooper, H.H. & Jacob, C.E. (1946). A generalized graphical method for evaluating formation constants and summarizing well-field history. *Trans. Am. Geophys. Union*, 27(4), 526–534.
- Jacob, C.E. (1947). Drawdown test to determine effective radius of artesian well. *Trans. ASCE*, 112, 1047–1064.
- Wyssling, L. (1979). Die Grundwasserschutzzonen der Wasserversorgungen. *SVGW*, Zürich.
- ACA — Agència Catalana de l'Aigua. Piezometric records and capture data. Sistema d'Informació del Medi Natural (SIMAN).
